# Basic RAG - chạy từng bước
Chạy lần lượt từng cell từ trên xuống.

In [ ]:
import json
import os
import warnings

from dotenv import load_dotenv
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore")
load_dotenv()

INDEX_DIR = "faiss_index"

## Bước 1: Load tài liệu từ web

In [ ]:
loader = WebBaseLoader("https://python.langchain.com/docs/get_started/introduction")
documents = loader.load()

print("Number of documents:", len(documents))
for i, doc in enumerate(documents):
    print(f"\nDocument {i + 1}")
    print("Metadata:", doc.metadata)
    print("Content length:", len(doc.page_content))
    print(doc.page_content[:1000])

## Bước 2: Chia chunk (1000 ký tự, gối đầu 200)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print("\nChunk đầu tiên:\n", chunks[0].page_content)

## Bước 3: Embedding + lưu vào FAISS

In [ ]:
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)

os.makedirs(INDEX_DIR, exist_ok=True)
vectorstore.save_local(INDEX_DIR)
print("Vectors in index:", vectorstore.index.ntotal)

## Bước 4: Xuất index.json để xem chunk
(Tuỳ chọn) Cell cuối của bước này có thể load lại index đã lưu để bỏ qua bước 1-3.

In [ ]:
metadata = {"chunks": []}
for i in range(vectorstore.index.ntotal):
    doc_id = vectorstore.index_to_docstore_id[i]
    doc = vectorstore.docstore.search(doc_id)
    metadata["chunks"].append({
        "id": str(doc_id),
        "text": doc.page_content,
        "source": doc.metadata.get("source", ""),
        "metadata": doc.metadata,
    })

with open(f"{INDEX_DIR}/index.json", "w") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)
print("Exported index.json")

In [ ]:
# (Tuỳ chọn) Load lại index đã lưu, bỏ qua bước 1-3
# vectorstore = FAISS.load_local(
#     INDEX_DIR, OpenAIEmbeddings(), allow_dangerous_deserialization=True
# )

## Bước 5: Retrieve - tìm chunk liên quan

In [ ]:
query = "What is LangChain?"  # đổi câu hỏi tại đây

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i + 1}:")
    print(doc.page_content[:500])

## Bước 6: Generate - LLM trả lời dựa trên chunk tìm được

In [ ]:
chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True,
)

response = chain.invoke({"query": query})
print("Sources:", len(response["source_documents"]))
print("Response:", response["result"])